# Step 0 · Data preparation

Builds the patent dataset of the flying-drones sector from PATSTAT (TIP):

1. **Search**: applications retrieved by keywords in titles and abstracts, IPC codes or CPC codes (OR).
2. **Family filter**: families with at least one patent of invention and a priority year in the window (section 1.1).
3. **SBERT test**: semantic similarity to the sector description; the calibration (section 2.3) found that a filter does not improve the dataset, so no threshold is applied.
4. **Output**: `df_selected.parquet`, the input of `3-DataExtractionAndCleaning`.

**SBERT runs on Colab**, because encoding about 150,000 texts is too slow on TIP. The notebook is therefore run in two parts and writes four files in total:

| Step | Where | File |
|---|---|---|
| Part A: section 1.1 | TIP | writes `df_family_filter.parquet` (the filter decision for every family, for the sensitivity check with utility models) |
| Part A: section 2.1 | TIP | writes `df_sbert_input.parquet` (applications to score) |
| `1c-SbertScoring_Colab.ipynb` | Colab | reads it, writes `df_sbert_scored.parquet` (the same applications with their score) |
| Part B: from section 2.2 | TIP | reads `df_sbert_scored.parquet`, writes `df_selected.parquet` (the dataset) |

Part B does not need part A in memory: after a kernel restart, run only the first three code cells (imports and settings) and continue from section 2.2.

This notebook produces the dataset; it does not judge it. Its precision is measured afterwards in `2c-ProdValidation.ipynb`, which reads `df_sbert_scored.parquet`.

How the keywords and classification codes were chosen is documented in `1a-QueryDiagnostics.ipynb`. The SBERT enrichment (adding applications from broader classes) was tested and dropped: see `validation/enrichment_decision.csv`.

In [1]:
from datetime import date

import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
%pip install -q pandas numpy sqlalchemy

from pathlib import Path

import numpy as np


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import yaml

# The choices that define the dataset are read from config/sector.yaml,
# the same file used by the other notebooks.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sector.yaml").exists())
cfg = yaml.safe_load((ROOT / "config" / "sector.yaml").read_text(encoding="utf-8"))

ENV = cfg["env"]                                                  # PROD for the final extraction
YEARS = (cfg["years"]["query_min"], cfg["years"]["query_max"])   # filing years searched by the query
FAMILY_YEARS = (cfg["years"]["min"], cfg["years"]["max"])         # priority years kept for families (section 1.1)
ipr = cfg["counting"]["ipr_type"]
KEEP_IPR_TYPES = (ipr,) if isinstance(ipr, str) else tuple(ipr)   # e.g. ("PI",), used in section 1.1
TAU_FILTER = float(cfg["sbert"]["tau_filter"])                    # -1.0 = no filter (calibration in section 2.3)

# Files of this notebook (df_family_filter.parquet is written in section 1.1):
SELECTED_FILE = Path(cfg["paths"]["df_selected"])           # 3. final dataset -> notebook 3
PROJECT_DIR = SELECTED_FILE.parent
SBERT_INPUT_FILE = PROJECT_DIR / "df_sbert_input.parquet"   # 1. applications to score     TIP -> Colab
SCORED_FILE = PROJECT_DIR / "df_sbert_scored.parquet"       # 2. the same, with their score Colab -> TIP

RANDOM_STATE = 42

print(f"ENV={ENV} | query years {YEARS} | family years {FAMILY_YEARS} | "
      f"IPR {KEEP_IPR_TYPES} | TAU_FILTER {TAU_FILTER}")

DRONE_DESCRIPTION = """
Flying drones: aircraft that fly without a human pilot on board, remotely
piloted or autonomous, of any size, configuration and end use.

In scope:
1. The vehicle: airframe, wings and rotors, propulsion and power supply,
   undercarriage.
2. Flight control, navigation, obstacle avoidance, command and data links,
   ground control stations, swarm coordination.
3. Dedicated infrastructure: launch, landing, recovery, charging or battery
   swap, docking, storage and transport.
4. Airspace and traffic management for unmanned aircraft.
5. Mission equipment and applications, when the invention concerns the
   aircraft itself or the way it is operated in flight (e.g. payload release
   for delivery, spraying controlled through the flight path).

Out of scope:
- Generic technologies for which a drone is only one of several possible
  carriers or uses.
- Inventions that only use data or services provided by a drone (e.g.
  analysis of images taken from a drone).
- Aircraft with a pilot on board.
- Ground-based counter-drone systems (a drone that intercepts other drones
  is in scope).
- Underwater, surface and ground unmanned vehicles.
- Guided missiles and projectiles.

Test for doubtful cases: if the drone is removed, does the invention still
make sense? If yes, it is out of scope.
""".strip()

ENV=PROD | query years (2000, 2024) | family years (2000, 2025) | IPR ('PI',) | TAU_FILTER -1.0


## Sector definition

Flying drones are aircraft that fly without a human pilot on board, remotely piloted or autonomous, of any size, configuration and end use.

| Area | In scope |
|---|---|
| **Vehicle** | Airframe, wings and rotors, propulsion and power supply, undercarriage |
| **Control and communications** | Flight control, navigation, obstacle avoidance, command and data links, ground control stations, swarms |
| **Dedicated infrastructure** | Launch, landing, recovery, charging or battery swap, docking, storage and transport |
| **Airspace management** | Traffic management for unmanned aircraft |
| **Mission equipment and applications** | Only when the invention concerns the aircraft itself or the way it is operated in flight |

Out of scope: generic technologies for which a drone is one of several possible uses; inventions that only use data or services provided by a drone; aircraft with a pilot on board; ground-based counter-drone systems; underwater, surface and ground unmanned vehicles; guided missiles and projectiles.

**Rule for doubtful cases:** if the drone is removed, does the invention still make sense? If yes, it is out of scope.

The full text is `DRONE_DESCRIPTION` above: it is the definition given to the LLM validator and to the human labellers. The similarity is computed against a shorter, positive-only version, `SBERT_DESCRIPTION`, defined in `1c-SbertScoring_Colab.ipynb`.

In [4]:
patstat = PatstatClient(env=ENV) # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()  # SQLAlchemy session used by all ORM queries below

## 1. Search: keywords, IPC and CPC

Final lists. Codes are matched by prefix (`B64U` selects the whole subclass). Keywords are matched as substrings of titles and abstracts, except "drone", which is matched as a whole word.

In [5]:
# Liste definitive. Come sono state scelte: Step0/1a-QueryDiagnostics.ipynb

DRONE_KEYWORDS = (
    # Names of the vehicle
    "unmanned aerial vehicle",
    "unmanned aerial system",
    "unmanned aircraft",
    "unmanned air vehicle",
    "unmanned helicopter",
    "uncrewed aircraft",
    "remotely piloted aircraft",

    # Configurations
    "quadcopter",
    "quadrotor",
    "quadrocopter",
    "multicopter",
    "hexacopter",
    "octocopter",
    "micro air vehicle",
)

# Cercate come parola intera (espressione regolare), non come pezzo di testo:
# "drone" come sottostringa trovava anche "dronedarone", "dihydronepetalactone", ...
DRONE_KEYWORD_PATTERNS = (
    r"\bdrones?\b",
)

DRONE_IPC_CODES = (
    "B64U",
    "G08G5/57",
)

DRONE_CPC_CODES = (
    "B64U",
    "B64C39/024",
    "G08G5/57",
)

In [6]:
def keyword_condition(terms, patterns=()):
    """OR-condition: a term (substring) or a pattern (regular expression)
    appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
        *(title.regexp_match(p) for p in patterns),
        *(abstract.regexp_match(p) for p in patterns),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Application-level fields retrieved by the search queries.
# These fields describe each application; they do not change the search criteria.
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,              # Unique application identifier
    TLS201_APPLN.docdb_family_id,       # Links applications belonging to the same DOCDB family
    TLS201_APPLN.appln_auth,            # Filing authority or patent office
    TLS201_APPLN.appln_nr,              # Application number
    TLS201_APPLN.appln_kind,            # Application kind code
    TLS201_APPLN.appln_filing_date,     # Filing date of this application
    TLS201_APPLN.earliest_filing_year,  # Earliest filing year associated with this application
    TLS201_APPLN.granted,               # Grant indicator: "Y" means granted
)

#3 TYPES OF QUERIES

#PROF BASELINE, MODIFIES:
#in each -> .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))

def query_by_keywords(*columns):
    """Applications whose title or abstract contains an Drone keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS, DRONE_KEYWORD_PATTERNS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )

# COMBINE THE SEARCH RESULTS

# Select application IDs found by at least one search strategy:
# keywords OR IPC codes OR CPC codes.
# UNION removes duplicate IDs across the three searches.

appln_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(appln_id)
    .union(
        query_by_ipc(appln_id),
        query_by_cpc(appln_id),
    )
    .subquery()
)

#each query (keyword,CPC,IPD) retrieve only appln_id foreach application

#with this function we retrieve all the others attributes
def query_details(ids_table):
    return (
        db.query(
            *APPLN_COLUMNS,
            TLS202_APPLN_TITLE.appln_title,
            TLS203_APPLN_ABSTR.appln_abstract,
        )
        .select_from(TLS201_APPLN)
        .join(
            ids_table,
            TLS201_APPLN.appln_id == ids_table.c.appln_id,
        )
        .outerjoin(
            TLS202_APPLN_TITLE,
            and_(
                TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
                func.lower(func.trim(TLS202_APPLN_TITLE.appln_title_lg)) == "en",
            ),
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            and_(
                TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
                func.lower(func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)) == "en",
            ),
        )
        .distinct()
    )

# Prepare the DataFrame with one row per patent application (appln_id).
# Replace missing titles and abstracts with empty strings.
# If multiple rows exist for the same application, keep the longest abstract.
def one_application(df):
    columns = [c.key for c in APPLN_COLUMNS]
    columns += ["appln_title", "appln_abstract"]
    df = df.reindex(columns=columns).copy()

    for column in ["appln_title", "appln_abstract"]:
        df[column] = df[column].fillna("").astype(str).str.strip()

    df["_abstract_length"] = df.appln_abstract.str.len()

    return (
        df.sort_values(
            ["appln_id", "_abstract_length", "appln_abstract", "appln_title"],
            ascending=[True, False, True, True],
        )
        .drop_duplicates("appln_id")
        .drop(columns="_abstract_length")
        .reset_index(drop=True)
    )

q_drone = query_details(drone_ids)
df_drone = one_application(patstat.df(q_drone))

df_drone.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,2442,38192468,EP,06838485,A,2006-11-29,2005,Y,ELECTRIC MOTOR ASSISTED TAKEOFF DEVICE FOR AN ...,"A takeoff assist device for an air vehicle, su..."
1,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...
2,28800,39609207,WO,2007069569,W,2007-05-23,2006,N,DUAL-USE MODULAR PROPULSION SURVEILLANCE VEHIC...,The present invention provides a system for re...
3,28822,39609212,WO,2007073846,W,2007-07-19,2006,N,AERODYNAMIC INTEGRATION OF A PAYLOAD CONTAINER...,A vertical takeoff and landing (VTOL) rotary-w...
4,36005,37810003,WO,2008000133,W,2008-01-16,2007,N,APPARATUS FOR A HIVE,A trap(1) for attachment to a hive entrance co...
5,39269,39636744,WO,2008051476,W,2008-01-18,2007,N,ROTARYCRAFT POWER AND PROPULSION SYSTEM,A rotary wing vehicle includes a body structur...
6,52470,38535646,EP,08101036,A,2008-01-29,2007,Y,Method and system for securely authorising the...,The system has a ground control station (100) ...
7,67606,39185865,FR,0608714,A,2006-10-03,2006,N,Electronic telemetry device for e.g. minidrone...,The device has a miniaturized controller (1) t...
8,68294,40210552,FR,0853393,A,2008-05-26,2008,Y,,
9,76326,39666849,WO,2007021962,W,2007-10-15,2006,N,LENTICULAR AIRSHIP,An airship (10) may include a hull (22) substa...


### 1.1 Patents of invention and priority years

Keeps the families that have at least one application of the types in `counting.ipr_type` (patents of invention, `PI`) and an earliest priority year within `years.min`–`years.max`. Utility models (`UM`, mostly Chinese, granted without examination on the merits) and designs (`DP`, which protect appearance only) are excluded at family level: a family stays if any of its members is a patent of invention. The rule looks at all members of each family in PATSTAT, not only those retrieved by the query.

The filter is applied before the export for SBERT, so only the kept families are scored. The decision for every family is saved in `df_family_filter.parquet`, for the sensitivity check with utility models.

In [8]:
FAMILY_FILTER_FILE = PROJECT_DIR / "df_family_filter.parquet"
CHUNK = 5000                                   # family ids per IN (...) list

valid_family = df_drone["docdb_family_id"].gt(0)
print(f"Applications without a valid family id (dropped): {int((~valid_family).sum()):,}")
df_drone = df_drone[valid_family].copy()
family_ids = sorted({int(f) for f in df_drone["docdb_family_id"]})

# All members of each family (not only the applications found by the query).
parts = []
for start in range(0, len(family_ids), CHUNK):
    chunk = family_ids[start:start + CHUNK]
    parts.append(patstat.df(
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.appln_id,
            TLS201_APPLN.ipr_type,
            TLS201_APPLN.earliest_filing_year,
        ).filter(TLS201_APPLN.docdb_family_id.in_(chunk))
    ))
members = pd.concat(parts, ignore_index=True)
members.columns = ["docdb_family_id", "appln_id", "ipr_type", "earliest_filing_year"]
members["ipr_type"] = members["ipr_type"].astype(str).str.strip()   # PATSTAT pads codes ("PI ")

# Priority year: earliest valid year among the members (9999 = unknown).
year = pd.to_numeric(members["earliest_filing_year"], errors="coerce")
members["valid_year"] = year.where(year.between(1800, 9998))
members["kept_type"] = members["ipr_type"].isin(KEEP_IPR_TYPES)

family_filter = members.groupby("docdb_family_id").agg(
    priority_year=("valid_year", "min"),
    has_kept_type=("kept_type", "any"),
    ipr_types=("ipr_type", lambda s: ",".join(sorted(set(s)))),
)
family_filter["in_years"] = family_filter["priority_year"].between(*FAMILY_YEARS)
family_filter["kept"] = family_filter["in_years"] & family_filter["has_kept_type"]
assert len(family_filter) == len(family_ids), "Some families were not found in TLS201"

print(f"Families retrieved: {len(family_filter):,}")
print(f"  outside priority years {FAMILY_YEARS}: {int((~family_filter['in_years']).sum()):,}")
print(f"  without a {'/'.join(KEEP_IPR_TYPES)} member: "
      f"{int((family_filter['in_years'] & ~family_filter['has_kept_type']).sum()):,}")
print(f"Families kept: {int(family_filter['kept'].sum()):,}")

# Save the decision for every family (sensitivity check with utility models).
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
family_filter.reset_index().to_parquet(FAMILY_FILTER_FILE, index=False)
print("Saved:", FAMILY_FILTER_FILE)

# From here on (SBERT export, calibration, df_selected) only the kept families are used.
df_drone = df_drone[df_drone["docdb_family_id"].isin(family_filter.index[family_filter["kept"]])].copy()
print(f"Applications in the kept families: {len(df_drone):,}")

Applications without a valid family id (dropped): 0
Families retrieved: 186,124
  outside priority years (2000, 2025): 37
  without a PI member: 67,639
Families kept: 118,448
Saved: /home/jovyan/progetto/df_family_filter.parquet
Applications in the kept families: 161,212


## 2. SBERT filter

Each application with an English abstract is compared with a short description of the sector; the cosine similarity is its score, and applications below `TAU_FILTER` are excluded.

Two choices follow from the validation of the first version of the filter: the text compared is **title + abstract** (many relevant applications name the drone only in the title), and the description contains **no exclusions** (an embedding model does not understand negation, so naming excluded technologies makes them more similar, not less).

### 2.1 Export for SBERT (end of part A)

Writes `df_sbert_input.parquet`: the applications that SBERT can score, with all their columns. Download it from the project folder and run `1c-SbertScoring_Colab.ipynb` on Colab.

Applications without an English abstract cannot be scored and are not part of the dataset: the cell prints how many they are, and how many families have no scorable application at all.

**Outcome:** the calibration in section 2.3 found that the filter does not improve the dataset, so it is not applied (`sbert.tau_filter = -1`). The scores are kept to document the test and for later steps.

In [9]:
#SECTION 2 : SBERT AS FILTER

# Applications with a non-empty English abstract: the only ones SBERT can score.
df_sbert = df_drone[
    df_drone["appln_abstract"].fillna("").str.strip().ne("")
].copy()

print("Applications with an abstract:", len(df_sbert))
print("Applications without an abstract:", len(df_drone) - len(df_sbert))

# Families that cannot be evaluated: none of their applications has an English abstract.
families_without_text = set(df_drone["docdb_family_id"]) - set(df_sbert["docdb_family_id"])
print("Families extracted:", df_drone["docdb_family_id"].nunique())
print("Families with no English abstract (not evaluated):", len(families_without_text))

# the applications to score, for 1c-SbertScoring_Colab.
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
df_sbert.to_parquet(SBERT_INPUT_FILE, index=False)
print("Saved:", SBERT_INPUT_FILE)

Applications with an abstract: 150487
Applications without an abstract: 10725
Families extracted: 118448
Families with no English abstract (not evaluated): 2957
Saved: /home/jovyan/progetto/df_sbert_input.parquet


### 2.2 Similarity scores (start of part B)

Reads `df_sbert_scored.parquet`, the file produced on Colab: upload it into the project folder before running. It contains the applications of `df_sbert_input.parquet` with one more column, `sbert_similarity`.

The check makes sure that the scores belong to this extraction and not to an earlier one.

In [12]:
# File 2 of 3: the scored applications.
df_sbert = pd.read_parquet(SCORED_FILE)

expected = pd.read_parquet(SBERT_INPUT_FILE, columns=["appln_id"])["appln_id"]
assert set(df_sbert["appln_id"]) == set(expected), (
    "The scored file does not match df_sbert_input.parquet: it comes from another extraction. "
    "Repeat the scoring on Colab with the current input file."
)

print("Applications with a score:", len(df_sbert))

# Applications most similar to the sector description.
df_sbert[
    ["appln_id", "appln_title", "sbert_similarity"]
].sort_values("sbert_similarity", ascending=False).head(10)

Applications with a score: 150487


,appln_id,appln_title,sbert_similarity
54742,526116352,COMBINATION OF UNMANNED AERIAL VEHICLES AND TH...,0.971339
22706,487106099,Flying system adopting large unmanned aerial v...,0.967252
100168,583001234,CONTROL SYSTEM AND METHOD FOR PREVENTING FLYIN...,0.966443
115865,600769122,UNMANNED AIRCRAFT SUPPORT DEVICE AND UNMANNED ...,0.966396
74905,550774678,AERIAL LAUNCH AND/OR RECOVERY FOR UNMANNED AIR...,0.965671
74906,550774723,AERIAL LAUNCH AND/OR RECOVERY FOR UNMANNED AIR...,0.965671
16062,477951970,/ AERIAL LAUNCH AND/OR RECOVERY FOR UNMANNED A...,0.965240
24626,489766292,Unmanned aerial vehicle flying system,0.965051
69482,544991619,Miniature aerial distribution type unmanned ae...,0.964581
86541,564864747,AERIAL LAUNCH AND RECOVERY OF UNMANNED AIRCRAF...,0.963789


### 2.3 Choice of the threshold

The table shows, for each candidate threshold, the share of relevant families kept (`relevant_kept`), the share of non-relevant families removed (`not_relevant_removed`), the precision of what is kept and the F1 score, on the families already labelled for the current sector definition. The first row (`tau = -1`) is the option without filter.

**Decision rule, fixed before looking at the results.** The metric is F1. Options whose F1 is within 0.01 of the best are a tie (with a few dozen non-relevant families in the sample, smaller differences are noise), and among ties the lowest threshold is chosen, because it loses fewer drones. The chosen value goes into `sbert.tau_filter` in `config/sector.yaml`; the table is saved in `validation/tau_filter_calibration.csv`.

The labelled families come from samples drawn by retrieval group in `1a-QueryDiagnostics`, so the noisiest group ("drone" keyword) is over-represented. This is acceptable: the sample is used only to choose the threshold, precision is measured afterwards on a new random sample (`2c-ProdValidation`), and the bias favours the filter (there is more noise to remove than in the pool), so if the filter does not beat "no filter" here, it would not on a random sample either.

In [14]:
# CALIBRATION OF TAU_FILTER
# Uses the families already labelled for the current sector definition
# (validation/llm_cache.jsonl, written by 1a-QueryDiagnostics). It makes no LLM calls.

%pip install -q langchain-core langchain-openai pydantic
%run -i ./2-DataValidation.ipynb

pid = prompt_id(DRONE_DESCRIPTION)
labels = pd.Series(
    {fam: r["llm_label"] for (fam, p), r in load_llm_cache().items() if p == pid},
    name="label", dtype="object",
)

# A family is kept if at least one of its applications reaches the threshold,
# so the family score is the highest similarity among its applications.
family_score = df_sbert.groupby("docdb_family_id")["sbert_similarity"].max().rename("score")

calibration = pd.concat([family_score, labels], axis=1, join="inner")
calibration = calibration[calibration["label"].isin(["pertinente", "non_pertinente"])]
relevant = calibration["label"].eq("pertinente")
print("Labelled families with a score:", len(calibration), "| relevant:", int(relevant.sum()))

# Decision rule, fixed before looking at the results: options whose F1 is within
# F1_TIE_MARGIN of the best are a tie; among ties the lowest threshold is chosen.
NO_FILTER = -1.0        # cosine similarity is always >= -1, so every family is kept
F1_TIE_MARGIN = 0.01

rows = []
for tau in np.r_[NO_FILTER, np.arange(0.0, 0.86, 0.05)]:
    tau = round(float(tau), 2)
    kept = calibration["score"].ge(tau)
    precision = (kept & relevant).sum() / max(kept.sum(), 1)
    recall = (kept & relevant).sum() / max(relevant.sum(), 1)
    rows.append({
        "tau": tau,
        "relevant_kept": round(recall, 3),
        "not_relevant_removed": round((~kept & ~relevant).sum() / max((~relevant).sum(), 1), 3),
        "precision_of_kept": round(precision, 3),
        "F1": round(2 * precision * recall / (precision + recall), 3) if precision + recall else 0.0,
    })
table = pd.DataFrame(rows)

tied = table[table["F1"] >= table["F1"].max() - F1_TIE_MARGIN]
chosen_tau = float(tied["tau"].min())
table["chosen"] = table["tau"].eq(chosen_tau)
table.to_csv(VALIDATION_DIR / "tau_filter_calibration.csv", index=False)

print("Chosen:", "no filter (tau = -1)" if chosen_tau == NO_FILTER else f"tau = {chosen_tau}")
if chosen_tau != TAU_FILTER:
    print(f"sbert.tau_filter in config/sector.yaml is {TAU_FILTER}: set it to {chosen_tau}.")
table


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Course API key:  ········


Labelled families with a score: 471 | relevant: 408
Chosen: no filter (tau = -1)


,tau,relevant_kept,not_relevant_removed,precision_of_kept,F1,chosen
0,-1.00,1.000,0.000,0.866,0.928,True
1,0.00,1.000,0.063,0.874,0.933,False
2,0.05,1.000,0.063,0.874,0.933,False
3,0.10,1.000,0.079,0.876,0.934,False
4,0.15,1.000,0.111,0.879,0.936,False
5,0.20,0.998,0.111,0.879,0.935,False
6,0.25,0.993,0.111,0.879,0.932,False
7,0.30,0.993,0.127,0.880,0.933,False
8,0.35,0.988,0.159,0.884,0.933,False
9,0.40,0.973,0.238,0.892,0.931,False


### 2.4 Filter

Applies `TAU_FILTER`. `df_selected` is the dataset.

In [15]:
# Select applications whose similarity reaches the filtering threshold.
df_filtered = df_sbert[
    df_sbert["sbert_similarity"] >= TAU_FILTER
].copy()

print("Applications evaluated with SBERT:", len(df_sbert))
print("Applications retained:", len(df_filtered))
print("Applications excluded by the filter:", len(df_sbert) - len(df_filtered))

df_filtered[
    ["appln_id", "appln_title", "sbert_similarity", "appln_abstract"]
].sort_values("sbert_similarity", ascending=False).head(10)

# Scartati dal filtro iniziale
df_rejected = df_sbert[
    df_sbert["sbert_similarity"] < TAU_FILTER
].copy()

Applications evaluated with SBERT: 150487
Applications retained: 150487
Applications excluded by the filter: 0


In [16]:
# Dataset produced by this notebook: the applications of the keyword/IPC/CPC union
# all scored applications (no SBERT filter).
df_selected = df_filtered.copy()
df_selected["retrieval_source"] = "initial_union"   # without sbert enrichment this column is useless

# Verify that each application appears only once.
assert not df_selected["appln_id"].duplicated().any()

print("Total selected applications:", len(df_selected))
print("DOCDB families:", df_selected["docdb_family_id"].nunique())

Total selected applications: 150487
DOCDB families: 115491


## 3. Output

`df_selected.parquet` is the dataset used by the following steps. To validate it, run `2c-ProdValidation.ipynb` with the same `TAU_FILTER` and `DRONE_DESCRIPTION`.

In [17]:
# Final dataset.
df_selected.to_parquet(SELECTED_FILE, index=False)

print(f"Dataset saved to: {SELECTED_FILE}")
print(f"Applications: {len(df_selected):,}")
print(f"DOCDB families: {df_selected['docdb_family_id'].nunique():,}")

Dataset saved to: /home/jovyan/progetto/df_selected.parquet
Applications: 150,487
DOCDB families: 115,491
